<a href="https://colab.research.google.com/github/LINWOO0099/Categorical-Encoding/blob/main/agent_queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math


# -----------------------------
# Mock Tool Implementations
# -----------------------------

def tool_calculator(action_input):
    """Evaluates safe math expressions using the math module."""
    try:
        allowed = {
            k: getattr(math, k)
            for k in dir(math)
            if not k.startswith("_")
        }

        return str(
            eval(
                action_input,
                {"__builtins__": {}},
                allowed
            )
        )

    except Exception as e:
        return f"Error: {e}"



def tool_unit_converter(action_input):
    """Converts km to miles."""
    try:
        parts = action_input.split()

        value = float(parts[0])

        return str(
            round(value * 0.621371, 4)
        )

    except Exception as e:
        return f"Error: {e}"



# -----------------------------
# Tool Registry
# -----------------------------

tool_registry = {

    "tool_calculator": {

        "name": "tool_calculator",

        "description":
        "Performs mathematical calculations: sqrt, log, sin, cos, abs, pi, round, power.",

        "function": tool_calculator
    },


    "tool_unit_converter": {

        "name": "tool_unit_converter",

        "description":
        "Converts between units of measurement.",

        "function": tool_unit_converter
    }
}



# -----------------------------
# Mock LLM
# -----------------------------

_mock_responses = [

"""Thought: I need to compute the square root of 144.
Action: tool_calculator
Action Input: sqrt(144)""",


"""Thought: Now I need to convert 10 km to miles.
Action: tool_unit_converter
Action Input: 10 km to miles""",


"""Thought: I have both results.
Final Answer: sqrt(144) = 12.0, and 10 km = 6.2137 miles."""
]


_call_count = [0]


def mock_llm(query, trace):

    idx = _call_count[0]

    _call_count[0] += 1

    if idx < len(_mock_responses):

        return _mock_responses[idx]

    return "Final Answer: No more responses."



# -----------------------------
# Parser Functions
# -----------------------------

def parse_thought(response):

    for line in response.splitlines():

        if line.startswith("Thought:"):

            return line.split(
                "Thought:",
                1
            )[1].strip()

    return ""



def parse_action(response):

    for line in response.splitlines():

        if line.startswith("Action:"):

            return line.split(
                "Action:",
                1
            )[1].strip()

    return ""



def parse_action_input(response):

    for line in response.splitlines():

        if line.startswith("Action Input:"):

            return line.split(
                "Action Input:",
                1
            )[1].strip()

    return ""



def parse_final_answer(response):

    for line in response.splitlines():

        if line.startswith("Final Answer:"):

            return line.split(
                "Final Answer:",
                1
            )[1].strip()

    return ""



# -----------------------------
# ReAct Agent Loop
# -----------------------------

def run_agent(query, tool_registry, max_steps=10):

    trace = []


    for step in range(max_steps):

        # Get LLM response
        response = mock_llm(query, trace)


        thought = parse_thought(response)

        action = parse_action(response)

        action_input = parse_action_input(response)

        final_answer = parse_final_answer(response)



        # Check final answer

        if final_answer:

            print(
                f"Thought: {thought}"
            )

            print(
                f"Final Answer: {final_answer}"
            )

            return final_answer



        # Execute tool

        if action in tool_registry:

            tool_function = tool_registry[action]["function"]

            observation = tool_function(action_input)

        else:

            observation = "Unknown tool"



        # Print step details

        print(
            f"Thought: {thought}"
        )

        print(
            f"Action: {action}"
        )

        print(
            f"Action Input: {action_input}"
        )

        print(
            f"Observation: {observation}"
        )



        # Store trace

        trace.append(

            {

                "thought": thought,

                "action": action,

                "action_input": action_input,

                "observation": observation

            }

        )



    return "Max steps reached without final answer."



# -----------------------------
# Run Agent
# -----------------------------

if __name__ == "__main__":

    result = run_agent(

        "Compute sqrt(144) and convert 10 km to miles.",

        tool_registry,

        max_steps=10

    )


    print("Result:", result)

Thought: I need to compute the square root of 144.
Action: tool_calculator
Action Input: sqrt(144)
Observation: 12.0
Thought: Now I need to convert 10 km to miles.
Action: tool_unit_converter
Action Input: 10 km to miles
Observation: 6.2137
Thought: I have both results.
Final Answer: sqrt(144) = 12.0, and 10 km = 6.2137 miles.
Result: sqrt(144) = 12.0, and 10 km = 6.2137 miles.
